# Summary Algorithm

In [12]:
# The summary algorithm computes a lot of descriptive statistics. I suggest to have a
# brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-summary-py/docs/v6-summary-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a two (federated-)step algorithm:
#
# 1. Call `summary_per_data_station`
# 2. Call `variance_per_data_station`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the summary statistics for
# the entire federated dataset. In IDEA4RC, the summary statistics per data station are
# also required. So in this notebook we go through the following steps to obtain both
# the *global* (from the central part) and the *local* (from the
# `summary_per_data_station` call) summary statistics:
#
# 1. Create a new vantage6 task to execute the *summary* method (central part). This
#    central part will start the tasks `summary_per_data_station` and
#    `variance_per_data_station` (as you can see in the swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* summary statistics from the central part (the main call)
# 4. Retrieve the *local* summary statistics from the data stations (the
#    `summary_per_data_station` call that was made by the central part)
#


In [13]:
import base64
import json
import requests

In [ ]:
headers = {
    "Authorization": "Bearer ***"
}

In [15]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [17]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATION_IDS = [org["id"] for org in response.json()["data"]]
ORGANIZATION_IDS

[1, 3]

In [18]:
ORGANIZATION_IDS = [1]

In [20]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 4
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 3

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "summary"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [89]

In [ ]:
# Before we can start analysis the cohorts (dataframes) we need to check the variables
# that are available in the dataframes. You can use this endpoint whenever you want user
# to allow you to select variables.
# TODO the dtpyes might change in the future, so do not rely on them to heavily now. In
# the next version of the data extraction job we will likely provide you with either the
# `category` or `numeric` colum type (so that you can use them to select varables)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{DATAFRAME_IDS[0]}",
    headers=headers
)
VARIABLES = response.json()["columns"]
VARIABLES

[{'name': 'patient_id', 'dtype': 'double', 'node_id': 7},
 {'name': 'age', 'dtype': 'double', 'node_id': 7},
 {'name': 'sex', 'dtype': 'string', 'node_id': 7},
 {'name': 'censor', 'dtype': 'bool', 'node_id': 7},
 {'name': 'status', 'dtype': 'string', 'node_id': 7},
 {'name': 'survival_days', 'dtype': 'null', 'node_id': 7},
 {'name': 'histology', 'dtype': 'string', 'node_id': 7},
 {'name': 'fnclcc_grade', 'dtype': 'string', 'node_id': 7},
 {'name': 'tumor_size', 'dtype': 'double', 'node_id': 7},
 {'name': 'surgery_date', 'dtype': 'double', 'node_id': 7},
 {'name': 'surgery_concept', 'dtype': 'double', 'node_id': 7},
 {'name': 'multifocality', 'dtype': 'string', 'node_id': 7},
 {'name': 'completeness_of_resection', 'dtype': 'string', 'node_id': 7},
 {'name': 'completeness_of_resection_concept_id',
  'dtype': 'double',
  'node_id': 7},
 {'name': 'tumor_rupture', 'dtype': 'double', 'node_id': 7},
 {'name': 'pre_operative_chemo', 'dtype': 'double', 'node_id': 7},
 {'name': 'post_operative_c

In [29]:
# For now lets hard code the variables that we want to include in the analysis.
VARIABLES = [
    "age", # num
    "tumor_size", # num
    "histology", # cat
    "sex", # cat
    "fnclcc_grade", # cat
    "multifocality", # cat
    "completeness_of_resection", # cat
    "tumor_rupture", # cat
    "pre_operative_chemo", # cat
    "post_operative_chemo", # cat
    "pre_operative_radio", # cat
    "post_operative_radio", # cat
    "local_recurrence", # cat
    "distant_metastasis", # cat
    "status", # cat
]
#
# From these variables, the numeric variables are:
NUMERIC_VARIABLES = [
    "age",
    "tumor_size"
]

In [33]:
org_input = [
    {
        "id": ORGANIZATION_IDS[0], # Central task
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "columns": VARIABLES,
                    "numeric_columns": NUMERIC_VARIABLES,
                    "organizations_to_include": ORGANIZATION_IDS # all participants
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

{'name': 'Human-readable name of the task',
 'image': 'harbor2.vantage6.ai/idea4rc/analytics:latest',
 'description': 'Description of the task',
 'action': 'central_compute',
 'method': 'summary',
 'organizations': [{'id': 1,
   'arguments': 'eyJjb2x1bW5zIjogWyJhZ2UiLCAidHVtb3Jfc2l6ZSIsICJoaXN0b2xvZ3kiLCAic2V4IiwgImZuY2xjY19ncmFkZSIsICJtdWx0aWZvY2FsaXR5IiwgImNvbXBsZXRlbmVzc19vZl9yZXNlY3Rpb24iLCAidHVtb3JfcnVwdHVyZSIsICJwcmVfb3BlcmF0aXZlX2NoZW1vIiwgInBvc3Rfb3BlcmF0aXZlX2NoZW1vIiwgInByZV9vcGVyYXRpdmVfcmFkaW8iLCAicG9zdF9vcGVyYXRpdmVfcmFkaW8iLCAibG9jYWxfcmVjdXJyZW5jZSIsICJkaXN0YW50X21ldGFzdGFzaXMiLCAic3RhdHVzIl0sICJudW1lcmljX2NvbHVtbnMiOiBbImFnZSIsICJ0dW1vcl9zaXplIl0sICJvcmdhbml6YXRpb25zX3RvX2luY2x1ZGUiOiBbMV19'}],
 'databases': [[{'type': 'dataframe', 'dataframe_id': 89}]],
 'session_id': 3,
 'study_id': 4}

In [34]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

{'study': {'id': 4,
  'link': '/server/study/4',
  'methods': ['PATCH', 'DELETE', 'GET']},
 'finished_at': None,
 'databases': [{'label': None,
   'type': 'dataframe',
   'dataframe_id': 89,
   'dataframe_name': 'modest_booth',
   'position': 0}],
 'id': 328,
 'results': '/server/result?task_id=328',
 'parent': None,
 'method': 'summary',
 'session': {'id': 3,
  'link': '/server/session/3',
  'methods': ['PATCH', 'DELETE', 'GET']},
 'created_at': '2025-12-10T14:17:41.671699',
 'required_by': [],
 'children': '/server/task?parent_id=328',
 'collaboration': {'id': 2,
  'link': '/server/collaboration/2',
  'methods': ['PATCH', 'DELETE', 'GET']},
 'status': 'awaiting',
 'name': 'Human-readable name of the task',
 'dataframe': None,
 'algorithm_store': None,
 'image': 'harbor2.vantage6.ai/idea4rc/analytics:latest',
 'action': 'central_compute',
 'init_org': {'id': 1,
  'link': '/server/organization/1',
  'methods': ['PATCH', 'DELETE', 'GET']},
 'job_id': 120,
 'init_user': {'id': 1,
  'link

In [60]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

'completed'

In [61]:
# Get the results of the (central) task, thus the *global* summary statistics.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

{'modest_booth': {'numeric': {'age': {'count': 10.0,
    'min': 42.0,
    'max': 71.0,
    'missing': 0.0,
    'sum': 570.0,
    'median': {'root': 57.5},
    'q_25': {'root': 49.75},
    'q_75': {'root': 64.75},
    'mean': 57.0,
    'std': 9.580071908799942},
   'tumor_size': {'count': 7.0,
    'min': 1.1,
    'max': 4.9,
    'missing': 3.0,
    'sum': 21.599999999999998,
    'median': {'root': 2.8},
    'q_25': {'root': 2.05},
    'q_75': {'root': 4.35},
    'mean': 3.0857142857142854,
    'std': 1.529083136423667}},
  'categorical': {'sex': {'count': 10, 'missing': 0},
   'completeness_of_resection': {'count': 10, 'missing': 0},
   'tumor_rupture': {'count': 10.0, 'missing': 0.0},
   'pre_operative_chemo': {'count': 10.0, 'missing': 0.0},
   'post_operative_chemo': {'count': 10.0, 'missing': 0.0},
   'local_recurrence': {'count': 10.0, 'missing': 0.0},
   'status': {'count': 10, 'missing': 0},
   'post_operative_radio': {'count': 10.0, 'missing': 0.0},
   'multifocality': {'count':

In [ ]:
# The central task created two subtasks, see the swimlane diagram (reference in the
# introduction of this notebook). We first need to retrieve the subtask IDs and then we
# can obtain the results from this. We do not need to poll until the subtasks are
# finished, as the central part is finished.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task?parent_id={TASK_ID}",
    headers=headers,
)
# We expect two subtasks for the summary algorithm, we are only interested in the one
# that triggers the `summary_per_data_station` method. I obtained the subtask ID here
# by looking at the method name, it is also possible to just obtain all the subtask IDs
# and then picking the lowest number (as `summary_per_data_station` is the first method
# to be called).
for task in response.json()["data"]:
    if task["method"] == "summary_per_data_station":
        SUBTASK_ID = task["id"]
        break
SUBTASK_ID

329

In [63]:
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={SUBTASK_ID}",
    headers=headers,
)
# Since this demo includes 2 organizations, we expect 2 results. Each for each center.
# The `organization_id` is included in the result, so we can easily identify the source.
for result in response.json()["data"]:
    print(json.loads(base64.b64decode(result["result"]).decode("UTF-8")))

{'modest_booth': {'numeric': {'age': {'count': 10.0, 'min': 42.0, 'max': 71.0, 'missing': 0.0, 'sum': 570.0, 'median': 57.5, 'q_25': 49.75, 'q_75': 64.75}, 'tumor_size': {'count': 7.0, 'min': 1.1, 'max': 4.9, 'missing': 3.0, 'sum': 21.599999999999998, 'median': 2.8, 'q_25': 2.05, 'q_75': 4.35}}, 'categorical': {'sex': {'count': 10, 'missing': 0}, 'completeness_of_resection': {'count': 10, 'missing': 0}, 'tumor_rupture': {'count': 10.0, 'missing': 0.0}, 'pre_operative_chemo': {'count': 10.0, 'missing': 0.0}, 'post_operative_chemo': {'count': 10.0, 'missing': 0.0}, 'local_recurrence': {'count': 10.0, 'missing': 0.0}, 'status': {'count': 10, 'missing': 0}, 'post_operative_radio': {'count': 10.0, 'missing': 0.0}, 'multifocality': {'count': 10, 'missing': 0}, 'distant_metastasis': {'count': 10.0, 'missing': 0.0}, 'fnclcc_grade': {'count': 10, 'missing': 0}, 'histology': {'count': 10, 'missing': 0}, 'pre_operative_radio': {'count': 10.0, 'missing': 0.0}}, 'num_complete_rows_per_node': 7, 'co